# Mini-Tritty: a tiny BitNet b1.58-style (ternary) language model on TinyStories

**How to use this notebook**
- Settings (right sidebar): **Accelerator = GPU**, **Internet = On**.
- Run the cells **top to bottom, one at a time**. Do not use "Run All" (the long training cells take hours).
- Every training run saves checkpoints to `/kaggle/working`. Download the backup zip (last cell) after each big step.
- New session after a restart? Attach your backup as a Kaggle Dataset (**Add Input**), run the first 3 code cells, then run the **Restore** cell, then continue from the cell you were on.

## 1. Hardware check and installs

In [ ]:
!nvidia-smi
!pip install -q tokenizers datasets

import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

## 2. Imports, settings, random seed

In [ ]:
import os, math, time, json, random, shutil, glob, contextlib
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reusable settings
device = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "/kaggle/working"   # files saved here survive until the session ends (download them!)
VOCAB_SIZE = 2048              # how many tokens the tokenizer knows
N_STORIES = 300_000            # stories used for the small experiments (full set is ~2.1M)

def autocast_ctx():
    """fp16 mixed precision on GPU, plain fp32 on CPU."""
    if device == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()

print("Using device:", device)

## 3. (Optional) Restore files from a backup Dataset

Only run this **after a session restart**, once your backup is attached via **Add Input**. It copies checkpoints, histories and the tokenizer back into `/kaggle/working`. If nothing is attached, it does nothing.

In [ ]:
restored = 0
for p in glob.glob("/kaggle/input/**/*", recursive=True):
    if os.path.isfile(p) and p.endswith((".pt", ".json", ".png")):
        shutil.copy(p, WORK_DIR)
        restored += 1
        print("restored", os.path.basename(p))
print(f"\nRestored {restored} files")
print(sorted(os.listdir(WORK_DIR)))

## 4. Load the TinyStories dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories", split=f"train[:{N_STORIES}]")
print(ds)

print("\n--- Example story ---")
print(ds[0]["text"])

sample = ds[:1000]["text"]
avg_chars = sum(len(t) for t in sample) / len(sample)
print(f"\nAverage story length: {avg_chars:.0f} characters")

## 5. Tokenizer (BPE)

If `tokenizer.json` already exists it is **loaded, not retrained**, so it always stays matched with your saved checkpoints.

In [ ]:
from tokenizers import ByteLevelBPETokenizer, Tokenizer

EOS = "<|endoftext|>"   # special token that marks "story ends here"
tok_path = f"{WORK_DIR}/tokenizer.json"

if os.path.exists(tok_path):
    print("tokenizer.json found: loading it (not retraining).")
else:
    def text_batches(batch_size=1000):
        for i in range(0, len(ds), batch_size):
            yield ds[i : i + batch_size]["text"]

    bpe = ByteLevelBPETokenizer()
    bpe.train_from_iterator(
        text_batches(),
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=[EOS],
        show_progress=True,
    )
    bpe.save(tok_path)

tok = Tokenizer.from_file(tok_path)
EOS_ID = tok.token_to_id(EOS)
print("Vocab size:", tok.get_vocab_size())
print("EOS id:", EOS_ID)

## 6. Test the tokenizer

In [ ]:
s = "Once upon a time, there was a little girl named Lily."
enc = tok.encode(s)

print("Tokens:", enc.tokens)
print("IDs:   ", enc.ids)
print("Decoded:", tok.decode(enc.ids))
print(f"\n{len(s)} characters -> {len(enc.ids)} tokens")

## 7. Tokenize the training stories into `train.bin`

Writes in chunks to a temporary file and renames it at the end, so an interrupted run can never leave a half-written `.bin` behind. Skips work if the file already exists.

In [ ]:
def tokenize_split(dataset, out_path, batch_size=2000, force=False):
    """Tokenize every story, add EOS after each, write one flat uint16 file. Returns token count."""
    if os.path.exists(out_path) and not force:
        n = os.path.getsize(out_path) // 2
        print(f"{os.path.basename(out_path)} already exists ({n:,} tokens), skipping. Use force=True to rebuild.")
        return n
    tmp_path = out_path + ".tmp"
    total, t0 = 0, time.time()
    with open(tmp_path, "wb") as f:
        for i in range(0, len(dataset), batch_size):
            texts = dataset[i : i + batch_size]["text"]
            ids = []
            for e in tok.encode_batch(texts):       # fast, parallel (written in Rust)
                ids.extend(e.ids)
                ids.append(EOS_ID)                  # mark "story ends here"
            assert max(ids) < VOCAB_SIZE, "token ID out of range!"
            f.write(np.array(ids, dtype=np.uint16).tobytes())
            total += len(ids)
            if (i // batch_size) % 25 == 0:
                print(f"  {i:>9,} / {len(dataset):,} stories  ({time.time()-t0:.0f}s)")
    os.replace(tmp_path, out_path)
    return total

n_train = tokenize_split(ds, f"{WORK_DIR}/train.bin")
print(f"\nTrain tokens: {n_train:,}")
print(f"File size: {os.path.getsize(f'{WORK_DIR}/train.bin') / 1e6:.0f} MB")

## 8. Tokenize the validation stories into `val.bin`

In [ ]:
val_ds = load_dataset("roneneldan/TinyStories", split="validation[:5000]")
n_val = tokenize_split(val_ds, f"{WORK_DIR}/val.bin")
print(f"\nValidation tokens: {n_val:,}")

## 9. Sanity check the files

In [ ]:
check = np.memmap(f"{WORK_DIR}/train.bin", dtype=np.uint16, mode="r")
print("Tokens in file:", len(check))
print("First 20 IDs:", check[:20].tolist())
print("\nDecoded:\n", tok.decode(check[:80].tolist()))
print("\nEOS tokens (= number of stories):", int((check == EOS_ID).sum()))

## 10. Model config

In [ ]:
@dataclass
class Config:
    vocab_size: int = VOCAB_SIZE   # 2048
    block_size: int = 256          # context length
    n_layer: int = 6               # number of transformer blocks
    n_head: int = 8                # attention heads
    dim: int = 256                 # width of the model
    hidden_dim: int = 704          # width inside the MLP
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    bitnet: bool = False           # False = normal weights, True = ternary

cfg = Config()
BATCH_SIZE = 64
print(cfg)

## 11. Training data batches

In [ ]:
def set_train_file(path):
    """Point get_batch('train') at a token file (train.bin or train_full.bin)."""
    global train_data
    train_data = np.memmap(path, dtype=np.uint16, mode="r")

set_train_file(f"{WORK_DIR}/train.bin")
val_data = np.memmap(f"{WORK_DIR}/val.bin", dtype=np.uint16, mode="r")

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - cfg.block_size - 1, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy(data[i : i + cfg.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + cfg.block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

# Test it: y must be x shifted left by one token
x, y = get_batch("train")
print("x shape:", x.shape, "| y shape:", y.shape)
print("\nx[0][:12]:", x[0][:12].tolist())
print("y[0][:12]:", y[0][:12].tolist())
print("\nx text:", tok.decode(x[0][:30].tolist()))

## 12. RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()   # do the math in fp32 for stability
        out = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.weight.float()).to(x.dtype)

# Test: a badly-scaled input should come out with RMS ~ 1
norm = RMSNorm(cfg.dim)
t = torch.randn(2, 5, cfg.dim) * 5
out = norm(t)
print("Input  RMS:", t.pow(2).mean(-1).sqrt()[0, :3].tolist())
print("Output RMS:", out.pow(2).mean(-1).sqrt()[0, :3].tolist())

## 13. RoPE (rotary position embeddings)

In [ ]:
def precompute_rope(head_dim, max_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))  # (hd/2,)
    t = torch.arange(max_len).float()                                            # positions
    angles = torch.outer(t, freqs)                                               # (max_len, hd/2)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin):
    # x: (batch, heads, time, head_dim). "Split in half" pairing: first half with second half.
    T = x.shape[2]
    cos = cos[:T][None, None].to(x.dtype)
    sin = sin[:T][None, None].to(x.dtype)
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

# Test: rotation must not change a vector's length
cos, sin = precompute_rope(cfg.dim // cfg.n_head, cfg.block_size, cfg.rope_theta)
q = torch.randn(1, cfg.n_head, 10, cfg.dim // cfg.n_head)
q_rot = apply_rope(q, cos, sin)
print("Same length before/after:", torch.allclose(q.norm(dim=-1), q_rot.norm(dim=-1), atol=1e-5))
print("But values changed:", not torch.allclose(q, q_rot))

## 14. BitLinear and the `make_linear` hook

This must come **before** the Attention and MLP cells, because they call `make_linear(..., norm=True)`.

In [ ]:
class BitLinear(nn.Linear):
    """Linear layer with ternary weights and int8 per-token activations (BitNet b1.58 style)."""
    def __init__(self, in_features, out_features, norm=False):
        super().__init__(in_features, out_features, bias=False)
        self.norm = RMSNorm(in_features) if norm else None

    @staticmethod
    def weight_quant(w):
        gamma = w.abs().mean().clamp(min=1e-5)          # one scale per matrix
        w_q = (w / gamma).round().clamp(-1, 1)          # values in {-1, 0, +1}
        return w_q, gamma

    @staticmethod
    def act_quant(x):
        s = 127.0 / x.abs().amax(dim=-1, keepdim=True).clamp(min=1e-5)   # per-token scale
        x_q = (x * s).round().clamp(-128, 127)          # integers
        return x_q, s

    def forward(self, x):
        if self.norm is not None:
            x = self.norm(x)
        w = self.weight                                  # latent fp32 weights
        w_q, gamma = self.weight_quant(w)
        x32 = x.float()
        x_q, s = self.act_quant(x32)

        # Straight-through estimator: forward = quantized, backward = identity
        w_ste = w   + (w_q * gamma - w).detach()
        x_ste = x32 + (x_q / s     - x32).detach()
        return F.linear(x_ste, w_ste)


def make_linear(in_features, out_features, cfg, norm=False):
    """Every weight matrix in the model is created here. cfg.bitnet decides the layer type."""
    if cfg.bitnet:
        return BitLinear(in_features, out_features, norm=norm)
    return nn.Linear(in_features, out_features, bias=False)

## 15. BitLinear test

In [ ]:
torch.manual_seed(0)
bl = BitLinear(256, 704).to(device)
nn.init.normal_(bl.weight, std=0.02)        # same init as our model
x = torch.randn(4, 16, 256, device=device)

# 1. Weight quantization
w_q, gamma = BitLinear.weight_quant(bl.weight)
print("Unique weight values:", sorted(w_q.unique().tolist()))
print(f"gamma: {gamma.item():.4f}")
print(f"zeros: {(w_q == 0).float().mean().item()*100:.1f}%   "
      f"+1: {(w_q == 1).float().mean().item()*100:.1f}%   "
      f"-1: {(w_q == -1).float().mean().item()*100:.1f}%")

# 2. Activation quantization
x_q, s = BitLinear.act_quant(x)
print("\nActivation int range:", x_q.min().item(), "to", x_q.max().item())
print("All integers:", bool((x_q == x_q.round()).all()))
print(f"Activation reconstruction error: {((x_q / s - x).norm() / x.norm()).item():.4f}")

# 3. Output vs the full-precision layer
y_bit = bl(x)
y_fp  = F.linear(x, bl.weight)
print(f"\nOutput relative error vs fp: {((y_bit - y_fp).norm() / y_fp.norm()).item():.3f}")

# 4. Gradients reach the latent weights
y_bit.sum().backward()
print("Gradient non-zero:", bl.weight.grad.abs().sum().item() > 0)

## 16. Attention

In [ ]:
class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.dim % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.dim // cfg.n_head
        self.wq = make_linear(cfg.dim, cfg.dim, cfg)
        self.wk = make_linear(cfg.dim, cfg.dim, cfg)
        self.wv = make_linear(cfg.dim, cfg.dim, cfg)
        self.wo = make_linear(cfg.dim, cfg.dim, cfg, norm=True)   # extra norm in front (ternary only)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        def split_heads(t):  # (B, T, C) -> (B, heads, T, head_dim)
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q = split_heads(self.wq(x))
        k = split_heads(self.wk(x))
        v = split_heads(self.wv(x))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)   # merge heads back
        return self.wo(y)

# Test (cfg.bitnet is False here, so these are plain Linear layers)
attn = Attention(cfg)
x_test = torch.randn(2, 16, cfg.dim)
print("Output shape:", attn(x_test, cos, sin).shape)
print(f"Attention params: {sum(p.numel() for p in attn.parameters()):,}   (expect 262,144)")

## 17. MLP (SwiGLU)

In [ ]:
class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = make_linear(cfg.dim, cfg.hidden_dim, cfg)
        self.w_up   = make_linear(cfg.dim, cfg.hidden_dim, cfg)
        self.w_down = make_linear(cfg.hidden_dim, cfg.dim, cfg, norm=True)   # extra norm in front (ternary only)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

mlp = MLP(cfg)
print("Output shape:", mlp(torch.randn(2, 16, cfg.dim)).shape)
print(f"MLP params: {sum(p.numel() for p in mlp.parameters()):,}   (expect 540,672)")

## 18. Transformer block

In [ ]:
class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.mlp = MLP(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)   # residual connection
        x = x + self.mlp(self.mlp_norm(x))               # residual connection
        return x

## 19. The full model, plus a size report for both versions

In [ ]:
class Transformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.final_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight          # weight tying

        cos, sin = precompute_rope(cfg.dim // cfg.n_head, cfg.block_size, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():            # smaller init for residual outputs
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x, self.rope_cos, self.rope_sin)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


def size_report(bitnet):
    cfg.bitnet = bitnet
    m = Transformer(cfg).to(device)
    n_total  = sum(p.numel() for p in m.parameters())
    n_linear = sum(mod.weight.numel() for n, mod in m.named_modules()
                   if isinstance(mod, nn.Linear) and n != "lm_head")
    n_emb    = m.tok_emb.weight.numel()
    n_bit    = sum(1 for mod in m.modules() if isinstance(mod, BitLinear))
    n_norm   = sum(1 for mod in m.modules() if isinstance(mod, BitLinear) and mod.norm is not None)
    print(f"[{'ternary' if bitnet else 'fp16 baseline'}]")
    print(f"  total params:          {n_total:,}   (expect {'5,350,272' if bitnet else '5,344,512'})")
    print(f"  block matrices:        {n_linear:,}")
    print(f"  embedding (tied):      {n_emb:,}")
    print(f"  BitLinear layers:      {n_bit}   (expect {'42' if bitnet else '0'})")
    print(f"  ...with inner norm:    {n_norm}   (expect {'12' if bitnet else '0'})")
    print(f"  fp32 size {n_total*4/1e6:.1f} MB | fp16 size {n_total*2/1e6:.1f} MB | "
          f"ternary est. {(n_linear*2/8 + n_emb)/1e6:.1f} MB (2-bit blocks + int8 embedding)")
    x, y = get_batch("train")
    with autocast_ctx():
        _, loss = m(x, y)
    print(f"  initial loss: {loss.item():.3f}   (random guessing = ln(2048) = {math.log(2048):.3f})\n")

size_report(False)
size_report(True)
cfg.bitnet = False

## 20. Training settings and learning-rate schedule

In [ ]:
WARMUP       = 200      # steps to ramp the learning rate up
WEIGHT_DECAY = 0.1
GRAD_CLIP    = 1.0
LOG_EVERY    = 100
EVAL_BATCHES = 40

def get_lr(step, max_steps, lr_max, lr_min):
    if step < WARMUP:                                   # linear warmup
        return lr_max * (step + 1) / WARMUP
    if step >= max_steps:
        return lr_min
    progress = (step - WARMUP) / (max_steps - WARMUP)   # cosine decay
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))

for s in [0, 100, 200, 1000, 3000, 5000, 5999]:
    print(f"step {s:>5}: lr = {get_lr(s, 6000, 1e-3, 1e-4):.6f}")
print(f"\nTokens per step: {BATCH_SIZE * cfg.block_size:,}")

## 21. Training function

`run_experiment` sets the data file, the layer type (`bitnet`) and the learning rate **itself**, so cell order can never silently change what gets trained. It checkpoints at every evaluation and **resumes automatically** if the checkpoint `<name>.pt` exists. Do not change `steps` for a run you want to resume (it changes the schedule).

In [ ]:
def make_optimizer(model, lr_max):
    decay    = [p for p in model.parameters() if p.dim() >= 2]   # matrices
    no_decay = [p for p in model.parameters() if p.dim() < 2]    # norm weights
    groups = [{"params": decay, "weight_decay": WEIGHT_DECAY},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=lr_max, betas=(0.9, 0.95), fused=(device == "cuda"))

@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(EVAL_BATCHES)
        for k in range(EVAL_BATCHES):
            x, y = get_batch(split)
            with autocast_ctx():
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def run_experiment(name, steps, lr, bitnet, data_file="train.bin", eval_every=500, resume=True):
    set_train_file(f"{WORK_DIR}/{data_file}")
    cfg.bitnet = bitnet
    lr_max, lr_min = lr, lr / 10

    torch.manual_seed(SEED)
    model  = Transformer(cfg).to(device)
    opt    = make_optimizer(model, lr_max)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    ckpt_path = f"{WORK_DIR}/{name}.pt"
    hist_path = f"{WORK_DIR}/{name}_history.json"
    step, best_val, history = 0, float("inf"), []

    if resume and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        step, best_val, history = ck["step"], ck["best_val"], ck["history"]
        print(f"Resumed '{name}' from step {step}")

    print(f"Run '{name}': {steps} steps, lr {lr_max:g} -> {lr_min:g}, "
          f"{'ternary' if bitnet else 'fp16'}, data {data_file}")
    model.train()
    t0 = time.time()
    while step < steps:
        lr_now = get_lr(step, steps, lr_max, lr_min)
        for g in opt.param_groups:
            g["lr"] = lr_now

        x, y = get_batch("train")
        with autocast_ctx():
            _, loss = model(x, y)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(opt)
        scaler.update()
        step += 1

        if step % LOG_EVERY == 0:
            dt = time.time() - t0
            tps = LOG_EVERY * x.numel() / dt
            print(f"step {step:>5} | loss {loss.item():.3f} | lr {lr_now:.5f} | "
                  f"gnorm {gnorm:.2f} | {tps/1000:.0f}k tok/s")
            t0 = time.time()

        if step % eval_every == 0 or step == steps:
            ev = estimate_loss(model)
            history.append({"step": step, "train": ev["train"], "val": ev["val"]})
            print(f"  >> EVAL step {step}: train {ev['train']:.3f} | val {ev['val']:.3f}")
            if ev["val"] < best_val:
                best_val = ev["val"]
                torch.save(model.state_dict(), f"{WORK_DIR}/{name}_best.pt")
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                        "scaler": scaler.state_dict(), "step": step,
                        "best_val": best_val, "history": history}, ckpt_path)
            with open(hist_path, "w") as f:
                json.dump(history, f)
            t0 = time.time()

    if history:
        print(f"\nDone '{name}'. Best val loss: {min(h['val'] for h in history):.4f}")
    return model, history

def best_val(name):
    p = f"{WORK_DIR}/{name}_history.json"
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return min(h["val"] for h in json.load(f))

## 22. Smoke tests (100 steps each, to catch bugs cheaply)

Expected: baseline loss ~4.3 at step 100, ternary a bit higher (~4.4 to 5.0). `nan` or a loss stuck near 7.6 means a bug.

In [ ]:
_, _ = run_experiment("smoke", 100, 1e-3, False, eval_every=100, resume=False)

In [ ]:
_, _ = run_experiment("smoke_bit", 100, 1e-3, True, eval_every=100, resume=False)

## 23. fp16 baseline (300k stories, 6000 steps, ~15 min on a T4)

Your earlier result for this exact setup: best val loss **1.667**.

In [ ]:
_, _ = run_experiment("baseline", 6000, 1e-3, False)

## 24. Ternary model (300k stories, 6000 steps, ~25 min on a T4)

Your earlier result for this exact setup: best val loss **1.849**.

In [ ]:
_, _ = run_experiment("ternary", 6000, 1e-3, True)

## 25. Plot: baseline vs ternary

In [ ]:
import matplotlib.pyplot as plt

def plot_runs(runs, title, out_name):
    plt.figure(figsize=(7.5, 4.5))
    for name, label, marker in runs:
        p = f"{WORK_DIR}/{name}_history.json"
        if not os.path.exists(p):
            print("skipping (no history yet):", name)
            continue
        with open(p) as f:
            h = json.load(f)
        plt.plot([r["step"] for r in h], [r["val"] for r in h], marker=marker, label=label)
    plt.xlabel("step"); plt.ylabel("validation loss"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig(f"{WORK_DIR}/{out_name}", dpi=150, bbox_inches="tight")
    plt.show()

plot_runs([("baseline", "fp16 baseline (val)", "s"),
           ("ternary", "ternary 1.58-bit (val)", "o")],
          "TinyStories, 5.3M params, 98M tokens", "comparison.png")

## 26. Text generation functions

In [ ]:
@torch.no_grad()
def generate2(model, prompt, max_new_tokens=200, temperature=0.7, top_k=40,
              rep_penalty=1.15, window=64):
    model.eval()
    ids = tok.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        x_cond = x[:, -cfg.block_size:]                      # only the last 256 tokens
        with autocast_ctx():
            logits, _ = model(x_cond)
        logits = logits[:, -1, :].float()                    # scores for the NEXT token
        recent = x[0, -window:].unique()                     # tokens used lately
        l = logits[0, recent]
        logits[0, recent] = torch.where(l > 0, l / rep_penalty, l * rep_penalty)
        logits = logits / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        if nxt.item() == EOS_ID:
            break
        x = torch.cat([x, nxt], dim=1)
    model.train()
    return tok.decode(x[0].tolist())

def generate(model, prompt, max_new_tokens=150, temperature=0.8, top_k=40):
    """Plain sampling (no repetition penalty), as used for the first comparison."""
    return generate2(model, prompt, max_new_tokens, temperature, top_k, rep_penalty=1.0)

def repeat_rate(text, n=3):
    w = text.split()
    grams = [tuple(w[i:i+n]) for i in range(len(w) - n + 1)]
    return 1 - len(set(grams)) / max(1, len(grams))

def load_model(path, bitnet):
    cfg.bitnet = bitnet            # decides which layer type gets built
    m = Transformer(cfg).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    return m

## 27. Same prompts, both models (plain sampling)

In [ ]:
base_m = load_model("/kaggle/input/datasets/haidarnasirodin/minitritty-modelsfp16-and-ternary/baseline_best.pt", False)
tern_m = load_model("/kaggle/input/datasets/haidarnasirodin/minitritty-modelsfp16-and-ternary/ternary_best.pt", True)

prompts = ["Once upon a time, there was a little girl named",
           "One day, a boy named Tom found a",
           "The dog was very sad because"]

for p in prompts:
    for label, m in [("FP16", base_m), ("TERNARY", tern_m)]:
        torch.manual_seed(1)            # same sampling randomness for both
        print(f"[{label}] {generate(m, p)}\n")
    print("=" * 70)

## 28. Better sampling: measure repetition for different settings

In [ ]:
base_m = load_model(f"{WORK_DIR}/baseline_best.pt", False)
tern_m = load_model(f"{WORK_DIR}/ternary_best.pt", True)

prompt = "One day, a boy named Tom found a"
settings = [(0.8, 1.0), (0.7, 1.0), (0.7, 1.15), (0.5, 1.1)]   # (temperature, rep_penalty)

for label, m in [("FP16", base_m), ("TERNARY", tern_m)]:
    print(f"=== {label} ===")
    for temp, rp in settings:
        torch.manual_seed(1)
        outs = [generate2(m, prompt, temperature=temp, rep_penalty=rp) for _ in range(10)]
        rr = sum(repeat_rate(o) for o in outs) / len(outs)
        print(f"temp {temp} | rep_penalty {rp} | repeated 3-grams: {rr*100:.1f}%")
    torch.manual_seed(1)
    print("\nSample (temp 0.7, penalty 1.15):\n", generate2(m, prompt), "\n")

## 29. Learning rate 2e-3: ternary (300k stories, ~25 min)

In [ ]:
_, _ = run_experiment("ternary_lr2e3", 6000, 2e-3, True)
print("\nTernary best val:  lr 1e-3 =", best_val("ternary"), "| lr 2e-3 =", best_val("ternary_lr2e3"))

## 30. Learning rate 2e-3: fp16 baseline (300k stories, ~15 min)

In [ ]:
_, _ = run_experiment("baseline_lr2e3", 6000, 2e-3, False)
print("\nBaseline best val: lr 1e-3 =", best_val("baseline"), "| lr 2e-3 =", best_val("baseline_lr2e3"))

## 31. Tokenize ALL 2.1M stories into `train_full.bin` (~10 min, ~1 GB)

In [ ]:
full = load_dataset("roneneldan/TinyStories", split="train")
print("Stories:", len(full))
n_full = tokenize_split(full, f"{WORK_DIR}/train_full.bin", batch_size=5000)
print(f"Tokens: {n_full:,}")
del full

## 32. Ternary long run on all data (30,000 steps, ~2 h on a T4)

Set `BEST_LR_TERNARY` from the learning-rate comparison above. Resumes automatically after a disconnect: restore your files, re-run the setup cells, then run this cell again.

In [ ]:
BEST_LR_TERNARY = 2e-3        # change to 1e-3 if the 2e-3 run was not clearly better

_, _ = run_experiment("ternary_full", 30000, BEST_LR_TERNARY, True,
                      data_file="train_full.bin", eval_every=1000)

## 33. fp16 baseline long run on all data (30,000 steps, ~1 h on a T4)

In [ ]:
BEST_LR_BASELINE = 1e-3       # change to 2e-3 if the 2e-3 run was clearly better

_, _ = run_experiment("baseline_full", 30000, BEST_LR_BASELINE, False,
                      data_file="train_full.bin", eval_every=1000)

## 34. Plot: long runs

In [ ]:
plot_runs([("baseline_full", "fp16 baseline, full data", "s"),
           ("ternary_full", "ternary, full data", "o")],
          "TinyStories, 5.3M params, 30k steps", "comparison_full.png")
print("baseline_full:", best_val("baseline_full"), "| ternary_full:", best_val("ternary_full"))

## 35. Backup everything that matters (then DOWNLOAD the zip!)

Output panel (right sidebar) -> `/kaggle/working` -> `minitritty_backup.zip` -> menu -> Download. Run this after every big step. Set `INCLUDE_RESUME = False` to skip the (bigger) resumable checkpoints.

In [ ]:
INCLUDE_RESUME = True

backup_dir = f"{WORK_DIR}/backup"
shutil.rmtree(backup_dir, ignore_errors=True)
os.makedirs(backup_dir)

for f in sorted(os.listdir(WORK_DIR)):
    src = f"{WORK_DIR}/{f}"
    if not os.path.isfile(src) or f.startswith("smoke") or f.endswith((".bin", ".tmp", ".zip")):
        continue
    is_resume_ckpt = f.endswith(".pt") and not f.endswith("_best.pt")
    if is_resume_ckpt and not INCLUDE_RESUME:
        continue
    if f.endswith((".pt", ".json", ".png")):
        shutil.copy(src, f"{backup_dir}/{f}")
        print(f"saved  {f}  ({os.path.getsize(src)/1e6:.1f} MB)")

zip_base = f"{WORK_DIR}/minitritty_backup"
shutil.make_archive(zip_base, "zip", backup_dir)
shutil.rmtree(backup_dir)
print(f"\nZip: {os.path.getsize(zip_base + '.zip')/1e6:.0f} MB  ->  {zip_base}.zip")

In [ ]:
import glob, zipfile
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    zipfile.ZipFile(z).extractall("/kaggle/working")
print(sorted(os.listdir("/kaggle/working")))

## Compare LR

In [ ]:
import json, glob
import matplotlib.pyplot as plt

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

runs = [("baseline_full_history.json", "fp16 baseline, full data"),
        ("ternary_full_history.json",  "ternary, full data")]

plt.figure(figsize=(7.5, 4.5))
for fn, label in runs:
    p = find(fn)
    if p is None:
        print("missing:", fn)
        continue
    h = json.load(open(p))
    print(f"{label}: best val {min(r['val'] for r in h):.4f} | last step {h[-1]['step']}")
    plt.plot([r["step"] for r in h], [r["val"] for r in h], marker=".", label=label)
plt.xlabel("step"); plt.ylabel("validation loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("TinyStories, 5.3M params, full data")
plt.savefig("/kaggle/working/comparison_full.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ternary_model = load_model("/kaggle/input/datasets/haidarnasirodin/minitritty-modelsfp16-and-ternary/ternary_best.pt", bitnet=True)
baseline_model = load_model("/kaggle/input/datasets/haidarnasirodin/minitritty-modelsfp16-and-ternary/baseline_best.pt", bitnet=False)

print(generate2(baseline_model, "One day, a little dog named"))
print(generate2(ternary_model, "One day, a little dog named"))